In [ ]:
import pandas as pd

# Paste your copied path inside the quotes
df = pd.read_csv('/content/drive/MyDrive/DataSet.csv')

In [ ]:
print("Shape:", df.shape)

print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

print("\nData Types:")
print(df.dtypes.value_counts())

print("\nTarget Distribution:")
print(df["F3924"].value_counts())

In [ ]:
missing_percent = (df.isnull().mean()*100)

print(
    missing_percent.sort_values(
        ascending=False
    ).head(30)
)

In [ ]:
full_missing = df.columns[df.isnull().all()]

print("Number of fully missing columns:")
print(len(full_missing))

In [ ]:
constant_cols = []

for col in df.columns:
    if df[col].nunique(dropna=False) == 1:
        constant_cols.append(col)

print("Constant Columns:")
print(len(constant_cols))

In [ ]:
missing_percent = df.isnull().mean()*100

print(missing_percent.describe())

In [ ]:
object_cols = df.select_dtypes(include=['object']).columns

print(object_cols)

for col in object_cols:
    print("\n")
    print(col)
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
df = df.drop(columns=full_missing)

In [ ]:
constant_cols = []

for col in df.columns:
    if df[col].nunique(dropna=False) == 1:
        constant_cols.append(col)

print("Constant Columns:")
print(len(constant_cols))

In [ ]:
df = df.drop(columns=constant_cols)

In [ ]:
fraud = df[df['F3924'] == 1]
normal = df[df['F3924'] == 0]

print("Fraud Accounts:", len(fraud))
print("Normal Accounts:", len(normal))

In [ ]:
pd.crosstab(
    df['F3886'],
    df['F3924'],
    normalize='index'
).sort_values(1, ascending=False)

In [ ]:
pd.crosstab(
    df['F3891'],
    df['F3924'],
    normalize='index'
).sort_values(1, ascending=False)

In [ ]:
pd.crosstab(
    df['F3893'],
    df['F3924'],
    normalize='index'
).sort_values(1, ascending=False)

In [ ]:
import numpy as np

corr_target = df.select_dtypes(include=np.number).corr()['F3924']

top_corr = corr_target.abs().sort_values(
    ascending=False
).head(30)

print(top_corr)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X = df.drop('F3924', axis=1)

# Temporarily encode object columns
X_encoded = X.copy()

for col in X_encoded.select_dtypes(include='object'):
    X_encoded[col] = X_encoded[col].astype('category').cat.codes

X_encoded = X_encoded.fillna(-999)

y = df['F3924']

mi = mutual_info_classif(
    X_encoded,
    y,
    random_state=42
)

mi_scores = pd.Series(
    mi,
    index=X_encoded.columns
)

print(
    mi_scores.sort_values(
        ascending=False
    ).head(30)
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_encoded, y)

importance = pd.Series(
    rf.feature_importances_,
    index=X_encoded.columns
)

print(
    importance.sort_values(
        ascending=False
    ).head(30)
)

In [ ]:
df.groupby("F3924")["F3912"].describe()

In [ ]:
df.groupby("F3924")["Unnamed: 0"].describe()

In [ ]:
pd.crosstab(
    df["F2230"],
    df["F3924"]
)

In [ ]:
df_clean = df.drop(columns=["Unnamed: 0"])

In [ ]:
df_clean = df_clean.drop(columns=["F3912"])

In [ ]:
df_clean = df_clean.drop(columns=["F2230"])

In [ ]:
print("Shape:", df.shape)

In [ ]:
X = df_clean.drop("F3924", axis=1)
y = df_clean["F3924"]

In [ ]:
print(
    X.select_dtypes(
        include='object'
    ).columns
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
print(
    X.select_dtypes(
        include='object'
    ).columns
)

In [ ]:
print(X.shape)
print(y.value_counts())

In [ ]:
missing_percent = (X.isnull().mean()*100)

print(
    missing_percent.sort_values(
        ascending=False
    ).head(20)
)

In [ ]:
missing_percent = X.isnull().mean() * 100

print("Columns > 99% missing:",
      (missing_percent > 99).sum())

print("Columns > 95% missing:",
      (missing_percent > 95).sum())

print("Columns > 90% missing:",
      (missing_percent > 90).sum())

print("Columns > 80% missing:",
      (missing_percent > 80).sum())

In [ ]:
high_missing_cols = missing_percent[
    missing_percent > 99
].index

for col in high_missing_cols[:10]:
    print("\n", col)
    print(pd.crosstab(
        df_clean[col].notna(),
        df_clean["F3924"]
    ))

In [ ]:
high_missing_cols = missing_percent[
    missing_percent > 99
].index

results = []

for col in high_missing_cols:

    present = df_clean[col].notna()

    fraud_present = df_clean.loc[present, "F3924"].mean()
    fraud_missing = df_clean.loc[~present, "F3924"].mean()

    results.append([
        col,
        present.sum(),
        fraud_present,
        fraud_missing,
        abs(fraud_present - fraud_missing)
    ])

audit_df = pd.DataFrame(
    results,
    columns=[
        "Feature",
        "Present_Count",
        "Fraud_When_Present",
        "Fraud_When_Missing",
        "Difference"
    ]
)

audit_df.sort_values(
    "Difference",
    ascending=False
).head(20)

In [ ]:
for col in ["F3287","F3290","F3611","F3614"]:

    print("\n", "="*50)
    print(col)

    print(df_clean[col].notna().value_counts())

    print("\nFraud distribution:")
    print(pd.crosstab(
        df_clean[col].notna(),
        df_clean["F3924"]
    ))

    print("\nUnique values:")
    print(df_clean[col].dropna().unique()[:20])

In [ ]:
important_features = [
    'F115','F321','F527','F531','F670',
    'F1692','F2082','F2122','F2582',
    'F2678','F2737','F2956','F3043',
    'F3836','F3887','F3889','F3891',
    'F3894'
]

for col in important_features:

    if col in df_clean.columns:

        print("\n" + "="*60)
        print(col)

        print("\nMissing %:")
        print(df_clean[col].isnull().mean()*100)

        print("\nUnique values:")
        print(df_clean[col].nunique())

        if df_clean[col].dtype == "object":
            print(df_clean[col].value_counts().head(10))
        else:
            print(df_clean[col].describe())

In [ ]:
important_features = [
    'F115','F321','F527','F531','F670',
    'F1692','F2082','F2122','F2582',
    'F2678','F2737','F2956','F3043',
    'F3836','F3887','F3894'
]

for col in important_features:

    print("\n" + "="*60)
    print(col)

    print(
        df_clean.groupby("F3924")[col]
        .describe()[["mean","50%","std"]]
    )

In [ ]:
from scipy.stats import mannwhitneyu

important_features = [
    'F115','F321','F527','F531','F670',
    'F1692','F2082','F2122','F2582',
    'F2678','F2737','F2956','F3043',
    'F3836','F3887','F3894'
]

results = []

for col in important_features:

    fraud = df_clean[df_clean["F3924"]==1][col].dropna()
    normal = df_clean[df_clean["F3924"]==0][col].dropna()

    try:
        stat, p = mannwhitneyu(
            fraud,
            normal,
            alternative='two-sided'
        )

        results.append([col,p])

    except:
        pass

pd.DataFrame(
    results,
    columns=["Feature","P_Value"]
).sort_values("P_Value")

Fraud Signal Features Identified

Tier-1:
F670
F115
F2082
F2122
F2956

Tier-2:
F1692

Suspected Leakage:
F3912
F2230
F3287
F3290
F3611
F3614
Unnamed:0

In [ ]:
important_features = [
    'F115',
    'F670',
    'F2082',
    'F2122',
    'F2956',
    'F1692'
]

for col in important_features:

    print("\n" + "="*60)
    print(col)

    print(df_clean.groupby("F3924")[col].quantile(
        [0,0.25,0.5,0.75,1]
    ))

In [ ]:
pd.crosstab(
    df_clean["F2082"] > 0,
    df_clean["F3924"]
)

In [ ]:
pd.crosstab(
    df_clean["F670"],
    df_clean["F3924"],
    normalize="index"
)

In [ ]:
df_clean["F2082"].value_counts().head(20)

F2082 > 0
⇒ Strong evidence account is legitimate

In [ ]:
df_clean.groupby("F3924")["F2082"].describe()

F2082

Observation

All fraud accounts have F2082 = 0.
No fraud account has F2082 > 0.

Conclusion

Extremely powerful fraud indicator.
Create binary feature:
F2082_nonzero.

Action

✅ Keep feature.

In [ ]:
df_clean["F2082_nonzero"] = (
    df_clean["F2082"] > 0
).astype(int)

In [ ]:
missing_percent = (
    df_clean.isnull().mean()*100
)

print(
    missing_percent.describe()
)

In [ ]:
print(
    missing_percent[
        missing_percent > 80
    ].shape
)

In [ ]:
high_missing = missing_percent[
    missing_percent > 95
].index

print(len(high_missing))

Remove only if BOTH are true:

Rule 1

Very high missing

>99%

AND

Rule 2

No relationship with fraud

Fraud_When_Present
≈
Fraud_When_Missing

In [ ]:
cols = [
    "F3287",
    "F3290",
    "F3611",
    "F3614"
]

for c in cols:
    print(c,
          df_clean[c].notna().sum())

In [ ]:
(
    df_clean[cols].notna()
).corr()

Derived features from same source
Instead of keeping all 4:

Keep:

F3287

Drop:

F3290
F3611
F3614



In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_num = df_clean.select_dtypes(
    include=["int64","float64"]
).drop(columns=["F3924"])

X_num = X_num.fillna(-999)

y = df_clean["F3924"]

mi = mutual_info_classif(
    X_num,
    y,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X_num.columns,
    "MI": mi
})

mi_df = mi_df.sort_values(
    "MI",
    ascending=False
)

print(mi_df.head(30))

Goal

Identify the most informative features in the entire dataset using Mutual Information.

Why

Correlation only captures linear relationships.

Fraud patterns are often nonlinear.

Mutual Information can detect hidden predictive signals.

Key findings:
1. F2082 extremely powerful.
2. F670 increases fraud rate ~3x.
3. F115 statistically significant.
4. F2956 statistically significant.
5. F3287 family is a rare fraud indicator.
6. 361 columns still have >95% missing.
7. Fraud class is highly imbalanced (0.89%).
8. F3912 shows suspiciously high correlation (0.969) and must be investigated before model training.

print(df_clean["F3912"].describe())

The feature was likely removed during preprocessing. This may be beneficial because such a highly predictive feature often indicates target leakage.

In [ ]:
"F3912" in df.columns

In [ ]:
"F3912" in df_clean.columns

In [ ]:
removed_cols = set(df.columns) - set(df_clean.columns)

print("F3912" in removed_cols)

In [ ]:
print(df["F3912"].isna().mean()*100)

print(df["F3912"].nunique())

print(pd.crosstab(df["F3912"], df["F3924"]))

F3912 is almost a perfect fraud indicator.
F3912=1 → 79 frauds out of 82 records
F3912=0 → 2 frauds out of 9000 records
Fraud Rates
F3912=1 → 96.34% fraud
F3912=0 → 0.022% fraud

In [ ]:
removed_cols = list(set(df.columns) - set(df_clean.columns))

print("F3912" in removed_cols)

F3912 is highly suspicious for target leakage or post-event information. Its removal during preprocessing is likely beneficial and should remain unless domain knowledge proves it is available before the fraud decision.

In [ ]:
df.shape
df_clean.shape

Rows      : 9082
Features  : 3857
Target    : F3924
Frauds    : 81
Non-frauds: 9001

In [ ]:
X = df_clean.drop("F3924", axis=1)
y = df_clean["F3924"]

In [ ]:
print(y.value_counts())
print(y.value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

In [ ]:
print(X_train.dtypes.value_counts())

In [ ]:
cat_cols = X_train.select_dtypes(include=["object"]).columns

print("Categorical columns:", len(cat_cols))
print(cat_cols[:20])

In [ ]:
for col in cat_cols:
    print("\n" + "="*50)
    print(col)
    print("Unique:", X_train[col].nunique())
    print(X_train[col].value_counts().head(10))

In [ ]:
X_train["F3888"] = pd.to_datetime(X_train["F3888"])
X_test["F3888"] = pd.to_datetime(X_test["F3888"])

In [ ]:
for df_ in [X_train, X_test]:
    df_["F3888_month"] = df_["F3888"].dt.month
    df_["F3888_day"] = df_["F3888"].dt.day
    df_["F3888_weekday"] = df_["F3888"].dt.weekday

In [ ]:
X_train.drop("F3888", axis=1, inplace=True)
X_test.drop("F3888", axis=1, inplace=True)

In [ ]:
missing = X_train.isnull().mean()*100

print(missing.describe())

In [ ]:
print((missing > 0).sum())

F3888 should be converted to datetime and decomposed into date components.
Remaining six categorical features are suitable for label encoding.
Dataset is nearly ready for modeling after missing-value assessment.

Missingness is widespread but highly uneven. Most features have little missing data, while a subset of features contains extremely large missing proportions. Because previous investigations identified highly predictive features with >99% missing values, missingness alone should not be used as a removal criterion.

In [ ]:
missing = X_train.isnull().mean()*100

print("Columns > 95% missing:", (missing > 95).sum())
print("Columns > 90% missing:", (missing > 90).sum())
print("Columns > 80% missing:", (missing > 80).sum())
print("Columns > 50% missing:", (missing > 50).sum())

X_train["F3888"] = pd.to_datetime(X_train["F3888"])
X_test["F3888"] = pd.to_datetime(X_test["F3888"])

for df_ in [X_train, X_test]:
    df_["F3888_month"] = df_["F3888"].dt.month
    df_["F3888_day"] = df_["F3888"].dt.day
    df_["F3888_weekday"] = df_["F3888"].dt.weekday

X_train.drop("F3888", axis=1, inplace=True)
X_test.drop("F3888", axis=1, inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = [
    "F3886",
    "F3889",
    "F3890",
    "F3891",
    "F3892",
    "F3893"
]

for col in cat_cols:
    le = LabelEncoder()

    combined = pd.concat([
        X_train[col].astype(str),
        X_test[col].astype(str)
    ])

    le.fit(combined)

    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

In [ ]:
cat_cols = X_train.select_dtypes(include=["object"]).columns
print(cat_cols)
print(len(cat_cols))

In [ ]:
print("F3888 in df_clean:", "F3888" in df_clean.columns)
print("F3888 in X:", "F3888" in X.columns)
print("F3888 in X_train:", "F3888" in X_train.columns)

In [ ]:
print(X_train.shape)

date_cols = [c for c in X_train.columns if c.startswith("F3888")]
print("Number of F3888 derived columns:", len(date_cols))

print(date_cols[:20])

In [ ]:
missing_train = X_train.isnull().sum().sum()
missing_test = X_test.isnull().sum().sum()

print("Train missing:", missing_train)
print("Test missing :", missing_test)

In [ ]:
missing_cols = X_train.columns[X_train.isnull().any()]

print("Columns with missing values:")
print(len(missing_cols))

In [ ]:
missing_pct_train = X_train.isnull().mean()*100

print("Columns >95% missing :", (missing_pct_train > 95).sum())
print("Columns >90% missing :", (missing_pct_train > 90).sum())
print("Columns >80% missing :", (missing_pct_train > 80).sum())
print("Columns >50% missing :", (missing_pct_train > 50).sum())
print("Columns >20% missing :", (missing_pct_train > 20).sum())

In [ ]:
high_missing = X_train.columns[
    X_train.isnull().mean() > 0.95
]

len(high_missing)

In [ ]:
results = []

for col in high_missing:
    present = X_train[col].notna()

    if present.sum() < 5:
        continue

    fraud_present = y_train[present].mean()
    fraud_missing = y_train[~present].mean()

    results.append([
        col,
        present.sum(),
        fraud_present,
        fraud_missing,
        abs(fraud_present - fraud_missing)
    ])

importance_df = pd.DataFrame(
    results,
    columns=[
        "Feature",
        "Present_Count",
        "Fraud_When_Present",
        "Fraud_When_Missing",
        "Difference"
    ]
)

importance_df.sort_values(
    "Difference",
    ascending=False
).head(30)

In [ ]:
dup_mask = X_train.T.duplicated()

duplicate_columns = X_train.columns[dup_mask]

print("Duplicate columns:", len(duplicate_columns))
print(duplicate_columns[:50])

In [ ]:
print("Before:")
print(X_train.shape)

X_train_nodup = X_train.loc[:, ~X_train.T.duplicated()]

print("\nAfter:")
print(X_train_nodup.shape)

print("\nRemoved:")
print(X_train.shape[1] - X_train_nodup.shape[1])

In [ ]:
X_train = X_train.loc[:, ~X_train.T.duplicated()]
X_test = X_test[X_train.columns]

print(X_train.shape)
print(X_test.shape)

25.8% reduction in dimensionality with zero information loss.



1. Target distribution	✅
2. Fraud imbalance analysis	✅
3. Missing value analysis	✅
4. High-missing feature analysis	✅
5. Duplicate feature detection	✅
6. Date feature engineering	✅
7. Leakage investigation	✅
8. Mutual Information analysis	✅




In [ ]:
for col in X_train.columns:
    if X_train[col].nunique(dropna=False) <= 5:
        tab = pd.crosstab(X_train[col], y_train)

        if 1 in tab.columns:
            max_fraud_rate = (
                tab[1] / tab.sum(axis=1)
            ).max()

            if max_fraud_rate > 0.90:
                print(col)
                print(tab)
                print("-"*50)

In [ ]:
missing_percent = X_train.isnull().mean()*100

print("Columns with missing values:",
      (missing_percent > 0).sum())

In [ ]:
missing_percent = X_train.isnull().mean()*100

print(missing_percent.describe())

print("Columns >95% missing :", (missing_percent > 95).sum())
print("Columns >90% missing :", (missing_percent > 90).sum())
print("Columns >80% missing :", (missing_percent > 80).sum())
print("Columns >50% missing :", (missing_percent > 50).sum())
print("Columns >20% missing :", (missing_percent > 20).sum())

Findings:

Missingness is highly skewed.
More than half the features have very little missing data.
A large subset of features (>220) contains less than 5% observed values.
Previous investigations showed that some highly sparse features are strong fraud indicators.

In [ ]:
cat_cols = X_train.select_dtypes(include=["object"]).columns

print("Categorical columns:", len(cat_cols))
print(cat_cols.tolist())

In [ ]:
for col in [
    "F3886",
    "F3889",
    "F3890",
    "F3891",
    "F3892",
    "F3893"
]:
    print(col, col in X_train.columns)

In [ ]:
constant_cols = []

for col in X_train.columns:
    if X_train[col].nunique(dropna=False) <= 1:
        constant_cols.append(col)

print("Constant columns:", len(constant_cols))
print(constant_cols[:50])

In [ ]:
X

In [ ]:
print("df_clean:", df_clean.shape)

print("X:", X.shape)

print("X_train:", X_train.shape)

print("X_test:", X_test.shape)

In [ ]:
for name, df_ in [
    ("df_clean", df_clean),
    ("X", X),
    ("X_train", X_train),
    ("X_test", X_test)
]:
    print("\n", name)

    cols = [
        c for c in [
            "F3886","F3889",
            "F3890","F3891",
            "F3892","F3893"
        ]
        if c in df_.columns
    ]

    print("Found:", cols)

    if len(cols):
        print(df_[cols].dtypes)

In [ ]:
fraud_ratio = y_train.mean()

print("Fraud ratio:", fraud_ratio)
print("Scale_pos_weight:", (len(y_train)-y_train.sum())/y_train.sum())

X["F3912"]

In [ ]:
X["F2082_nonzero"] = (X["F2082"] > 0).astype(int)

In [ ]:
X_train = X_train.fillna(-999)
X_test = X_test.fillna(-999)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=110.77,
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
pred_prob = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import average_precision_score

ap = average_precision_score(y_test, pred_prob)

print(ap)

In [ ]:
from sklearn.metrics import classification_report

pred = (pred_prob > 0.5).astype(int)

print(classification_report(y_test, pred))

After handling:

Duplicate columns
Constant columns
Date features
Categorical encoding
Class imbalance using scale_pos_weight

an XGBoost model was trained. The model separates fraudulent and non-fraudulent transactions extremely well. Only one fraud case was missed in the validation set. The feature engineering and cleaning steps significantly improved signal quality.

Metric	Value
Accuracy	~99.95%
Fraud Precision	1.00
Fraud Recall	0.94
Fraud F1	0.97
Average Precision	0.9901

In [ ]:
import pandas as pd

pd.Series(pred_prob).describe()

In [ ]:
pd.Series(pred_prob).quantile(
    [0.90,0.95,0.99,0.995,0.999]
)

In [ ]:
imp = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

imp = imp.sort_values(
    "Importance",
    ascending=False
)

print(imp.head(30))

Develop additional models:

XGBoost
LightGBM
CatBoost

and compare cross-validation performance.

Conclusion

Ensembling or selecting the strongest boosting model may provide the final performance improvement needed for a top leaderboard position.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X, y):

    X_tr = X.iloc[train_idx]
    X_val = X.iloc[valid_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[valid_idx]

    model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=111,
        random_state=42,
        eval_metric='logloss'
    )

    model.fit(X_tr, y_tr)

    pred_prob = model.predict_proba(X_val)[:,1]

    score = average_precision_score(
        y_val,
        pred_prob
    )

    scores.append(score)

print(scores)
print("Mean AP:", np.mean(scores))

In [ ]:
print(X.dtypes[X.dtypes == "object"])

In [ ]:
for var in dir():
    obj = globals()[var]

    if isinstance(obj, pd.DataFrame):
        print(var, obj.shape)

In [ ]:
X.dtypes[X.dtypes == "object"]

Verify whether the observed AP score generalizes across different splits.

Method

Perform Stratified 5-Fold Cross Validation using Average Precision.

In [ ]:
X_final = X_train_nodup.copy()

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_final, y_train):

    X_tr = X_final.iloc[train_idx]
    X_val = X_final.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_tr, y_tr)

    pred_prob = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(y_val, pred_prob)

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.9345257306795767), np.float64(0.8228937728937726), np.float64(0.7980303918352645), np.float64(0.9999999999999998), np.float64(0.8201272411798727)]
Mean AP: 0.8751154273176972


In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_final.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

importance_df.head(50)

,Feature,Importance
1269,F1821,0.040640
139,F160,0.036289
1734,F2390,0.022559
1435,F2030,0.021848
1522,F2137,0.020574
1174,F1707,0.019134
383,F450,0.017176
910,F1231,0.016814
2882,F3897,0.015246
1376,F1961,0.013070


In [ ]:
top_features = importance_df.head(20)["Feature"].tolist()

for col in top_features:
    print("\n", col)

    print(
        pd.crosstab(
            X_final[col],
            y_train
        ).head(20)
    )


 F1821
F3924     0  1
F1821         
0.00    474  0
5.96      1  0
20.00     1  0
40.00     1  0
100.00    2  0
118.00    2  0
130.00    1  0
200.00    2  0
236.00    1  0
300.00    1  0
329.00    1  0
350.00    1  0
362.00    4  0
432.00    1  0
500.00    7  0
620.00    1  0
652.00    1  0
700.00    6  0
744.00    1  0
750.00    1  0

 F160
F3924    0  1
F160         
0.00   162  0
0.01     6  0
0.02     6  0
0.03     2  0
0.04     4  1
0.05     5  0
0.06     5  0
0.07     4  0
0.08     5  0
0.09     4  0
0.10     2  0
0.11     6  0
0.12     3  0
0.13     8  0
0.14     8  0
0.15     6  0
0.16     2  0
0.17    12  0
0.18     8  0
0.19     6  0

 F2390
F3924    0   1
F2390         
0.00   323  20
0.01    66   1
0.02    58   2
0.03    49   2
0.04    48   2
0.05    37   0
0.06    30   1
0.07    39   0
0.08    39   0
0.09    36   0
0.10    31   1
0.11    32   1
0.12    41   1
0.13    30   0
0.14    33   1
0.15    30   1
0.16    31   0
0.17    32   0
0.18    29   2
0.19    36   0

 F2030
F

In [ ]:
top_features = [
    'F1821','F160','F2390','F2030','F2137',
    'F1707','F450','F1231','F3897','F1961'
]

for col in top_features:
    print("\n", col)

    grp = df.groupby("F3924")[col].agg(
        ['mean','median','min','max']
    )

    print(grp)


 F1821
               mean     median      min           max
F3924                                                
0      4.488447e+07  730000.00     0.00  6.234266e+10
1      3.228267e+05  126234.51  2419.87  4.476267e+06

 F160
           mean  median   min  max
F3924                             
0      0.627603   0.600  0.00  1.0
1      0.798125   0.905  0.04  1.0

 F2390
            mean  median  min     max
F3924                                
0       2.332558    0.65  0.0  7172.8
1      29.931646    0.14  0.0  2311.0

 F2030
                mean       median  min           max
F3924                                               
0      382874.720362  9906.349261  0.0  6.515907e+08
1        5255.080539  1913.989899  0.0  1.299636e+05

 F2137
                mean        median         min           max
F3924                                                       
0      729354.293172  20247.430570    0.000000  1.305649e+09
1        8811.547944   3817.023913  529.992778  1.126732e+

In [ ]:
for col in ["F3896","F3897","F3923"]:
    print("\n", col)

    tab = pd.crosstab(
        df[col],
        df["F3924"],
        normalize="index"
    )

    print(tab.tail(20))


 F3896
F3924         0         1
F3896                    
400    1.000000  0.000000
500    1.000000  0.000000
600    0.990481  0.009519
650    0.978723  0.021277
700    0.992278  0.007722
750    0.823529  0.176471
800    0.968750  0.031250
900    0.994737  0.005263

 F3897
F3924         0         1
F3897                    
0      0.991264  0.008736
1      0.992571  0.007429
2      0.986726  0.013274
3      0.953488  0.046512
4      1.000000  0.000000
5      1.000000  0.000000
6      1.000000  0.000000
7      1.000000  0.000000
13     1.000000  0.000000
15     1.000000  0.000000

 F3923
F3924         0         1
F3923                    
0      0.991527  0.008473
1      0.991892  0.008108
2      0.969697  0.030303
3      0.869565  0.130435
4      1.000000  0.000000
5      1.000000  0.000000
6      1.000000  0.000000


In [ ]:
duplicates = df.duplicated().sum()
print("Duplicates:", duplicates)

Duplicates: 0


In [ ]:
fraud_duplicates = (
    df[df["F3924"]==1]
    .duplicated()
    .sum()
)

print("Fraud duplicates:", fraud_duplicates)

Fraud duplicates: 0




*  No duplicate fraud rows
*  No feature with 100% fraud rate
*  Cross-validation remains strong (0.875 AP)
*  Performance collapses when most features are removed





In [ ]:
top20 = importance_df.head(20)["Feature"].tolist()

X_top20 = X_final[top20]

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top20, y_train):

    X_tr = X_top20.iloc[train_idx]
    X_val = X_top20.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_tr, y_tr)

    pred_prob = model.predict_proba(X_val)[:, 1]

    ap = average_precision_score(y_val, pred_prob)

    scores.append(ap)

print("Fold AP scores:", scores)
print("Mean AP:", np.mean(scores))

Fold AP scores: [np.float64(0.7078683011065595), np.float64(0.48202148440528786), np.float64(0.23186464567592824), np.float64(0.48238572300548777), np.float64(0.4492844870921157)]
Mean AP: 0.4706849282570758


Your model is not being driven by only a handful of features.

Instead, XGBoost is learning from:

hundreds or thousands of weak signals,
interactions between features,
nonlinear combinations

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_pred = np.zeros(len(y_train))

for train_idx, val_idx in skf.split(X_final, y_train):

    X_tr = X_final.iloc[train_idx]
    X_val = X_final.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_tr, y_tr)

    oof_pred[val_idx] = model.predict_proba(X_val)[:,1]

oof_ap = average_precision_score(y_train, oof_pred)

print("OOF AP:", oof_ap)

OOF AP: 0.8774775683941202


5-Fold Mean AP: 0.8751
OOF AP: 0.8775

The OOF score matching the CV score almost exactly is a good sign that your evaluation is stable and not being inflated by a particular split.

In [ ]:
import pandas as pd

pd.Series(oof_pred).describe()

,0
count,7265.000000
mean,0.007806
std,0.078513
min,0.000016
25%,0.000078
50%,0.000140
75%,0.000328
max,0.999958


In [ ]:
pd.Series(oof_pred).quantile(
    [0.90,0.95,0.99,0.995,0.999]
)

,0
0.900,0.000984
0.950,0.002603
0.990,0.092981
0.995,0.934111
0.999,0.999712


In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(
    y_train,
    oof_pred
)

f1 = 2 * precision * recall / (precision + recall + 1e-10)

best_idx = np.argmax(f1)

print("Best threshold:", thresholds[best_idx])
print("Precision:", precision[best_idx])
print("Recall:", recall[best_idx])
print("F1:", f1[best_idx])

Best threshold: 0.30496466159820557
Precision: 0.9636363636363636
Recall: 0.8153846153846154
F1: 0.8833333332836806


In [ ]:
final_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=110.77,
    eval_metric="logloss",
    random_state=42
)

final_model.fit(X_final, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

test_prob = final_model.predict_proba(X_test_final)[:, 1]

In [ ]:
print(X_final.shape)
print(X_test.shape)

(7265, 2911)
(1817, 2911)


In [ ]:
print(len(X_final.columns))
print(len(X_test.columns))

2911
2911


In [ ]:
missing = set(X_final.columns) - set(X_test.columns)
extra = set(X_test.columns) - set(X_final.columns)

print("Missing:", len(missing))
print("Extra:", len(extra))

Missing: 0
Extra: 0


In [ ]:
test_prob = final_model.predict_proba(X_test)[:, 1]

In [ ]:
import pandas as pd

pd.Series(test_prob).describe()

,0
count,1817.000000
mean,0.000497
std,0.005280
min,0.000019
25%,0.000076
50%,0.000112
75%,0.000180
max,0.134428


In [ ]:
print(X_final.shape)
print(X_test.shape)
print(len(X_final.columns))
print(len(X_test.columns))

(7265, 2911)
(1817, 2911)
2911
2911


In [ ]:
sum(test_prob > 0.30496466159820557)

np.int64(0)

In [ ]:
submission = pd.DataFrame({
    "Fraud_Prob": test_prob
})

submission.sort_values(
    "Fraud_Prob",
    ascending=False
).head(20)

,Fraud_Prob
55,0.134428
1496,0.120162
1283,0.094152
1460,0.072250
1273,0.038851
217,0.027203
740,0.025063
1103,0.023953
831,0.019599
997,0.016595


submission = pd.DataFrame({
    "ID": test_ids,
    "F3924": test_prob
})

submission.to_csv(
    "submission.csv",
    index=False
)

In [ ]:
print(df_.head())

       F1   F2   F3   F4   F5   F6    F7   F8   F9   F10  ...  F3918  F3919  \
6557  NaN  NaN  NaN  NaN  NaN  NaN   NaN  NaN  NaN   NaN  ...      0      2   
8972  NaN  NaN  NaN  NaN  NaN  NaN   NaN  NaN  NaN   NaN  ...      0      1   
5690  0.0  0.0  NaN  0.0  0.0  NaN  0.00  0.0  0.0  0.00  ...      0      1   
3327  NaN  NaN  NaN  NaN  NaN  NaN   NaN  NaN  NaN   NaN  ...      0      1   
3495  1.0  NaN  1.0  1.0  NaN  1.0  0.67  1.0  0.5  0.87  ...      0      1   

      F3920  F3921  F3922  F3923  F2082_nonzero  F3888_month  F3888_day  \
6557      1      1      0      0              0           12          1   
8972      1      0      0      0              0            1          7   
5690      0      1      0      0              0           12         31   
3327      0      0      0      0              0           12         28   
3495      0      0      1      0              1           10          6   

      F3888_weekday  
6557              3  
8972              1  
5690    

In [ ]:
important_bank_features = [
    "F115","F321","F527","F531","F670",
    "F1692","F2082","F2122","F2582",
    "F2678","F2737","F2956","F3043",
    "F3836","F3887","F3889","F3891","F3894"
]

importance_df[
    importance_df["Feature"].isin(
        important_bank_features
    )
].sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
2874,F3889,0.003042
277,F321,0.002282
548,F670,0.000315
1902,F2582,0.000210
2879,F3894,0.000155
2168,F2956,0.000115
2876,F3891,0.000100
2873,F3887,0.000052
455,F527,0.000048
2040,F2737,0.000005


In [ ]:
pd.crosstab(
    df["F3889"],
    y_train,
    normalize="index"
).sort_values(1, ascending=False).head(20)

F3924,0,1
F3889,,
L365D,0.983607,0.016393
G365D,0.990582,0.009418
L31D,0.990909,0.009091
L180D,0.991935,0.008065
L14D,1.000000,0.000000
L7D,1.000000,0.000000
L90D,1.000000,0.000000


In [ ]:
df.groupby("F3924")["F3889"].describe()

,count,unique,top,freq
F3924,,,,
0,9001,7,G365D,7472
1,81,4,G365D,72


In [ ]:
top20 = importance_df.head(20)["Feature"].tolist()

In [ ]:
for col in top20:
    fraud_mean = df.loc[df["F3924"]==1, col].mean()
    normal_mean = df.loc[df["F3924"]==0, col].mean()

    ratio = abs(fraud_mean - normal_mean)

    print(col, ratio)

F1821 44561641.44606568
F160 0.17052243028179292
F2390 27.599087348643188
F2030 377619.6398232785
F2137 720542.7452276903
F1707 43929910.71598029
F450 0.9638962472406181
F1231 538.6305136784747
F3897 0.09085958898942642
F1961 0.050022190415327454
F210 0.18476526185311426
F2835 91.77697841726619
F20 0.0774752091235491
F3896 10.998366436651168
F3085 54.70464601769913
F411 0.07572280756120242
F3923 0.13220478931696203
F228 0.17293328017012233
F2486 1.0957828962831149
F2837 96.01626016260163


from sklearn.metrics import average_precision_score

for col in top20:

    ap = average_precision_score(
        y_train,
        df.loc[y_train.index, col]
    )

    print(col, ap)

In [ ]:
for col in top20:
    n_missing = df[col].isna().sum()
    if n_missing > 0:
        print(col, n_missing)

F1821 4
F160 2153
F2390 3292
F2030 7
F2137 4
F1707 7
F450 7264
F1231 2
F1961 13
F210 7231
F2835 8524
F20 1007
F3085 7268
F411 7231
F228 641
F2486 3348
F2837 8588


In [ ]:
for col in top20:

    temp = df.loc[y_train.index, col]

    ap = average_precision_score(
        y_train,
        temp.fillna(temp.median())
    )

    print(col, round(ap, 4))

F1821 0.0054
F160 0.0235
F2390 0.0075
F2030 0.0053
F2137 0.0052
F1707 0.0054
F450 0.009
F1231 0.0111
F3897 0.0116
F1961 0.0075
F210 0.009
F2835 0.0089
F20 0.0122
F3896 0.0104
F3085 0.0096
F411 0.009
F3923 0.0178
F228 0.0149
F2486 0.0068
F2837 0.0089


In [ ]:
for col in ["F3896", "F3897", "F3923"]:

    print("\n", col)

    temp = pd.crosstab(
        df[col],
        df["F3924"],
        normalize="index"
    )

    print(temp.sort_values(1, ascending=False).head(20))


 F3896
F3924         0         1
F3896                    
750    0.823529  0.176471
800    0.968750  0.031250
650    0.978723  0.021277
600    0.990481  0.009519
700    0.992278  0.007722
900    0.994737  0.005263
400    1.000000  0.000000
500    1.000000  0.000000

 F3897
F3924         0         1
F3897                    
3      0.953488  0.046512
2      0.986726  0.013274
0      0.991264  0.008736
1      0.992571  0.007429
4      1.000000  0.000000
5      1.000000  0.000000
6      1.000000  0.000000
7      1.000000  0.000000
13     1.000000  0.000000
15     1.000000  0.000000

 F3923
F3924         0         1
F3923                    
3      0.869565  0.130435
2      0.969697  0.030303
0      0.991527  0.008473
1      0.991892  0.008108
4      1.000000  0.000000
5      1.000000  0.000000
6      1.000000  0.000000


In [ ]:
pd.crosstab(
    [df["F3896"], df["F3923"]],
    df["F3924"],
    normalize="index"
).sort_values(1, ascending=False).head(30)

,F3924,0,1
F3896,F3923,,
750,3,0.000000,1.000000
800,2,0.500000,0.500000
600,2,0.916667,0.083333
900,1,0.916667,0.083333
750,0,0.933333,0.066667
700,3,0.944444,0.055556
650,0,0.977778,0.022222
600,0,0.990552,0.009448
700,0,0.992198,0.007802


In [ ]:
high_missing = [
    "F450",
    "F210",
    "F411",
    "F3085",
    "F2835",
    "F2837",
    "F518"
]

for col in high_missing:
    df[col + "_missing"] = df[col].isna().astype(int)

In [ ]:
df["missing_count"] = df.isna().sum(axis=1)

df.groupby("F3924")["missing_count"].describe()

,count,mean,std,min,25%,50%,75%,max
F3924,,,,,,,,
0,9001.0,1020.919676,119.085253,636.0,943.0,1013.0,1086.0,2267.0
1,81.0,1057.061728,100.784094,767.0,1007.0,1067.0,1135.0,1287.0


In [ ]:
df["missing_ratio"] = df.isna().mean(axis=1)

In [ ]:
X_final["F3896_F3923"] = (
    X_final["F3896"].astype(str)
    + "_"
    + X_final["F3923"].astype(str)
)

In [ ]:
X_final["F3896_F3897"] = (
    X_final["F3896"].astype(str)
    + "_"
    + X_final["F3897"].astype(str)
)

In [ ]:
X_final["F3923_F3897"] = (
    X_final["F3923"].astype(str)
    + "_"
    + X_final["F3897"].astype(str)
)

In [ ]:
risk_map = y_train.groupby(X_final["F3923"]).mean()

In [ ]:
X_final["F3923_risk"] = X_final["F3923"].map(risk_map)

pd.qcut(
    train_df["F160"],
    q=10,
    duplicates="drop"
).to_frame().join(y_train).groupby("F160")["F3924"].mean()

In [ ]:
print(type(X_final))
print(X_final.shape)
print(X_final.columns[:10])

<class 'pandas.core.frame.DataFrame'>
(7265, 2915)
Index(['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10'], dtype='object')


In [ ]:
pd.qcut(
    X_final["F160"],
    q=10,
    duplicates="drop"
).to_frame().join(y_train).groupby("F160")["F3924"].mean()

/tmp/ipykernel_1689/1522641124.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).to_frame().join(y_train).groupby("F160")["F3924"].mean()


,F3924
F160,
"(-0.001, 0.33]",0.008945
"(0.33, 0.43]",0.001686
"(0.43, 0.49]",0.000000
"(0.49, 0.54]",0.008282
"(0.54, 0.61]",0.011457
"(0.61, 0.69]",0.003953
"(0.69, 0.79]",0.009025
"(0.79, 0.9]",0.015355
"(0.9, 1.0]",0.029091


In [ ]:
for col in ["F160","F228","F518","F3923","F3896","F3897"]:
    tmp = pd.DataFrame({
        col: X_final[col],
        "target": y_train.values
    })

    try:
        tmp["bin"] = pd.qcut(tmp[col], q=10, duplicates="drop")
        print("\n", col)
        print(tmp.groupby("bin")["target"].agg(["count","mean"]))
    except:
        print("\n", col)
        print(tmp.groupby(col)["target"].agg(["count","mean"]).sort_values("mean", ascending=False).head(20))


 F160
                count      mean
bin                            
(-0.001, 0.33]    559  0.008945
(0.33, 0.43]      593  0.001686
(0.43, 0.49]      610  0.000000
(0.49, 0.54]      483  0.008282
(0.54, 0.61]      611  0.011457
(0.61, 0.69]      506  0.003953
(0.69, 0.79]      554  0.009025
(0.79, 0.9]       521  0.015355
(0.9, 1.0]       1100  0.029091

 F228
                count      mean
bin                            
(-0.001, 0.18]    741  0.005398
(0.18, 0.24]      695  0.001439
(0.24, 0.29]      669  0.002990
(0.29, 0.35]      655  0.006107
(0.35, 0.43]      658  0.003040
(0.43, 0.53]      660  0.007576
(0.53, 0.67]      675  0.022222
(0.67, 0.86]      653  0.012251
(0.86, 1.0]      1337  0.017951

 F518
                 count      mean
bin                             
(-0.001, 0.089]    488  0.000000
(0.089, 1.166]      70  0.000000
(1.166, 2.592]      70  0.000000
(2.592, 13.23]      70  0.014286

 F3923
               count      mean
bin                           
(-0.001

/tmp/ipykernel_1689/2673560888.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("bin")["target"].agg(["count","mean"]))
/tmp/ipykernel_1689/2673560888.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("bin")["target"].agg(["count","mean"]))
/tmp/ipykernel_1689/2673560888.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("bin")["target"].agg(["count","mean"])

In [ ]:
X_final["F160_high"] = (X_final["F160"] > 0.9).astype(int)
X_test["F160_high"] = (X_test["F160"] > 0.9).astype(int)

In [ ]:
X_final["F228_high"] = (X_final["F228"] > 0.53).astype(int)
X_test["F228_high"] = (X_test["F228"] > 0.53).astype(int)

In [ ]:
X_final["F3896_highrisk"] = (X_final["F3896"] >= 750).astype(int)
X_test["F3896_highrisk"] = (X_test["F3896"] >= 750).astype(int)

In [ ]:
X_final["F3897_high"] = (X_final["F3897"] >= 2).astype(int)
X_test["F3897_high"] = (X_test["F3897"] >= 2).astype(int)

In [ ]:
X_final["risk_combo_1"] = (
    (X_final["F3896"] >= 750) &
    (X_final["F3897"] >= 2)
).astype(int)

X_test["risk_combo_1"] = (
    (X_test["F3896"] >= 750) &
    (X_test["F3897"] >= 2)
).astype(int)

In [ ]:
X_final["risk_combo_2"] = (
    (X_final["F160"] > 0.9) &
    (X_final["F228"] > 0.53)
).astype(int)

X_test["risk_combo_2"] = (
    (X_test["F160"] > 0.9) &
    (X_test["F228"] > 0.53)
).astype(int)

In [ ]:
X_final["risk_combo_3"] = (
    (X_final["F3923"] >= 2)
).astype(int)

X_test["risk_combo_3"] = (
    (X_test["F3923"] >= 2)
).astype(int)

In [ ]:
for col in [
    "F160_high",
    "F228_high",
    "F3896_highrisk",
    "F3897_high",
    "risk_combo_1",
    "risk_combo_2",
    "risk_combo_3"
]:
    print("\n", col)
    print(
        pd.crosstab(
            X_final[col],
            y_train,
            normalize="index"
        )
    )


 F160_high
F3924             0         1
F160_high                    
0          0.994647  0.005353
1          0.970909  0.029091

 F228_high
F3924             0         1
F228_high                    
0          0.996087  0.003913
1          0.982364  0.017636

 F3896_highrisk
F3924                  0         1
F3896_highrisk                    
0               0.991524  0.008476
1               0.973118  0.026882

 F3897_high
F3924              0         1
F3897_high                    
0           0.991559  0.008441
1           0.978182  0.021818

 risk_combo_1
F3924                0         1
risk_combo_1                    
0             0.991401  0.008599
1             0.945455  0.054545

 risk_combo_2
F3924                0         1
risk_combo_2                    
0             0.994366  0.005634
1             0.966857  0.033143

 risk_combo_3
F3924                0         1
risk_combo_3                    
0             0.991654  0.008346
1             0.934211  0.065789


In [ ]:
new_features = [
    "F160_high",
    "F228_high",
    "F3896_highrisk",
    "F3897_high",
    "risk_combo_1",
    "risk_combo_2",
    "risk_combo_3"
]

In [ ]:
X_eng = X_final.copy()

In [ ]:
for c in [
    "F160_high",
    "F228_high",
    "F3896_highrisk",
    "F3897_high",
    "risk_combo_1",
    "risk_combo_2",
    "risk_combo_3"
]:
    print(c, c in X_eng.columns)

F160_high True
F228_high True
F3896_highrisk True
F3897_high True
risk_combo_1 True
risk_combo_2 True
risk_combo_3 True


from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_eng, y_train):

    X_tr = X_eng.iloc[train_idx]
    X_val = X_eng.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_tr, y_tr)

    pred_prob = model.predict_proba(X_val)[:, 1]

    ap = average_precision_score(y_val, pred_prob)

    scores.append(ap)

print("Fold AP scores:", scores)
print("Mean AP:", np.mean(scores))

In [ ]:
X_eng[[
    "F3896_F3923",
    "F3896_F3897",
    "F3923_F3897"
]].dtypes

,0
F3896_F3923,object
F3896_F3897,object
F3923_F3897,object


In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in [
    "F3896_F3923",
    "F3896_F3897",
    "F3923_F3897"
]:
    le = LabelEncoder()
    X_eng[col] = le.fit_transform(X_eng[col].astype(str))

In [ ]:
X_eng[[
    "F3896_F3923",
    "F3896_F3897",
    "F3923_F3897"
]].dtypes

,0
F3896_F3923,int64
F3896_F3897,int64
F3923_F3897,int64


In [ ]:
print(X_eng.shape)

(7265, 2922)


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_eng, y_train):

    X_tr = X_eng.iloc[train_idx]
    X_val = X_eng.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_tr, y_tr)

    pred_prob = model.predict_proba(X_val)[:, 1]

    ap = average_precision_score(y_val, pred_prob)

    scores.append(ap)

print("Fold AP scores:", scores)
print("Mean AP:", np.mean(scores))

Fold AP scores: [np.float64(0.9407935369473828), np.float64(0.8228451828218454), np.float64(0.8008317873888298), np.float64(0.9897435897435896), np.float64(0.837477407554198)]
Mean AP: 0.8783383008911692


The fact that it is almost identical to your OOF AP (~0.8775) means:

✅ The engineered features are highly informative
✅ The model is learning strong fraud patterns
✅ The validation process is now much more consistent

In [ ]:
for col in [
    "F160_high",
    "F228_high",
    "F3896_highrisk",
    "F3897_high",
    "risk_combo_1",
    "risk_combo_2",
    "risk_combo_3",
    "expert_risk_score",
    "F3896_F3923",
    "F3896_F3897",
    "F3923_F3897"
]:
    print(col, col in y_train.index.astype(str))

F160_high False
F228_high False
F3896_highrisk False
F3897_high False
risk_combo_1 False
risk_combo_2 False
risk_combo_3 False
expert_risk_score False
F3896_F3923 False
F3896_F3897 False
F3923_F3897 False


In [ ]:
print(X_eng.shape)
print(y_train.shape)

(7265, 2922)
(7265,)


In [ ]:
X_eng.columns[-15:]

Index(['F2082_nonzero', 'F3888_month', 'F3888_day', 'F3888_weekday',
       'F3896_F3923', 'F3896_F3897', 'F3923_F3897', 'F3923_risk', 'F160_high',
       'F228_high', 'F3896_highrisk', 'F3897_high', 'risk_combo_1',
       'risk_combo_2', 'risk_combo_3'],
      dtype='object')

In [ ]:
final_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=110.77,
    eval_metric="logloss",
    random_state=42
)

final_model.fit(X_eng, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_eng.columns,
    "Importance": final_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

importance_df.head(30)

,Feature,Importance
1173,F1706,0.028919
1744,F2400,0.022198
1041,F1491,0.019680
1350,F1921,0.018058
1088,F1598,0.017366
2793,F3806,0.016802
1258,F1810,0.016491
120,F141,0.014619
2883,F3898,0.014137
1089,F1599,0.013730


pd.crosstab(
    X_eng["expert_risk_score"],
    y_train,
    normalize="index"
).sort_index()

In [ ]:
X_eng["expert_risk_score"] = (
    X_eng["F160_high"]
    + X_eng["F228_high"]
    + X_eng["F3896_highrisk"]
    + X_eng["F3897_high"]
)

In [ ]:
pd.crosstab(
    X_eng["expert_risk_score"],
    y_train,
    normalize="index"
).sort_index()

F3924,0,1
expert_risk_score,,
0,0.997091,0.002909
1,0.990080,0.009920
2,0.969979,0.030021
3,0.980392,0.019608
4,0.666667,0.333333


top50 = importance_df.head(50)["Feature"].tolist()

for col in top50:

    if X_eng[col].nunique() <= 20:

        print("\n", col)

        print(
            pd.crosstab(
                X_eng[col],
                y_train,
                normalize="index"
            ).sort_values(1, ascending=False).head(10)
        )

In [ ]:
for col in [
    "F2400",
    "F1706",
    "F1599",
    "F1491",
    "F1921"
]:

    print("\n", col)

    print(
        X_eng[col].describe()
    )

    print(
        X_eng.groupby(y_train)[col]
        .median()
    )


 F2400
count     1189.000000
mean        15.788823
std        408.120626
min          0.000000
25%          0.000000
50%          0.260000
75%          1.050000
max      13876.260000
Name: F2400, dtype: float64
F3924
0    0.26
1    0.02
Name: F2400, dtype: float64

 F1706
count    7.260000e+03
mean     5.656289e+07
std      1.536004e+09
min      0.000000e+00
25%      1.969550e+05
50%      5.881455e+05
75%      1.665595e+06
max      8.242865e+10
Name: F1706, dtype: float64
F3924
0    597540.53
1     76756.00
Name: F1706, dtype: float64

 F1599
count    7.255000e+03
mean     2.348306e+07
std      5.477243e+08
min      0.000000e+00
25%      1.143935e+05
50%      3.804000e+05
75%      1.078813e+06
max      2.420136e+10
Name: F1599, dtype: float64
F3924
0    388624.02
1     81987.18
Name: F1599, dtype: float64

 F1491
count    7.263000e+03
mean     7.064826e+06
std      1.518291e+08
min      0.000000e+00
25%      3.694250e+04
50%      1.000000e+05
75%      4.957460e+05
max      7.687900e+0

In [ ]:
X_eng["F3920_highrisk"] = X_eng["F3920"].isin([2,3,4]).astype(int)

X_eng["F3921_highrisk"] = X_eng["F3921"].isin([2,3,8]).astype(int)

X_eng["F3898_highrisk"] = X_eng["F3898"].isin([1,2]).astype(int)

In [ ]:
risk_features = [
    "F160_high",
    "F228_high",
    "F3896_highrisk",
    "F3897_high",
    "F3920_highrisk",
    "F3921_highrisk",
    "F3898_highrisk"
]

X_eng["super_risk_score"] = X_eng[risk_features].sum(axis=1)

In [ ]:
pd.crosstab(
    X_eng["super_risk_score"],
    y_train,
    normalize="index"
).sort_index()

F3924,0,1
super_risk_score,,
0,0.998814,0.001186
1,0.994200,0.005800
2,0.981667,0.018333
3,0.907950,0.092050
4,0.944444,0.055556
5,0.750000,0.250000


In [ ]:
num_cols = [
    "F2400",
    "F1706",
    "F1599",
    "F1491",
    "F1921"
]

for col in num_cols:

    temp = pd.DataFrame({
        col: X_eng[col],
        "target": y_train
    })

    temp["bin"] = pd.qcut(
        temp[col],
        q=10,
        duplicates="drop"
    )

    print("\n", col)

    print(
        temp.groupby("bin")["target"]
        .agg(["count","mean"])
    )


 F2400
                  count      mean
bin                              
(-0.001, 0.07]      479  0.006263
(0.07, 0.26]        117  0.000000
(0.26, 0.56]        119  0.000000
(0.56, 0.89]        120  0.008333
(0.89, 1.28]        117  0.000000
(1.28, 2.22]        119  0.000000
(2.22, 13876.26]    118  0.000000

 F1706
                            count      mean
bin                                        
(-0.001, 57843.3]             726  0.038567
(57843.3, 148234.8]           726  0.016529
(148234.8, 250012.6]          726  0.012397
(250012.6, 390095.18]         726  0.009642
(390095.18, 588145.475]       726  0.002755
(588145.475, 820887.648]      726  0.001377
(820887.648, 1228752.744]     726  0.004132
(1228752.744, 2532332.794]    726  0.002755
(2532332.794, 8327822.0]      726  0.001377
(8327822.0, 82428651510.0]    726  0.000000

 F1599
                              count      mean
bin                                          
(-0.001, 32959.8]               726  0.015152
(329

/tmp/ipykernel_1689/536225410.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp.groupby("bin")["target"]
/tmp/ipykernel_1689/536225410.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp.groupby("bin")["target"]
/tmp/ipykernel_1689/536225410.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp.groupby("bin")["target"]
/tmp/ipykernel_1689/536225410.py:25: FutureWarning: The default of observed=False is 

X_eng["F2400_highrisk"] = (
    X_eng["F2400"] > threshold
).astype(int)

In [ ]:
print(X_eng.shape)

(7265, 2927)


In [ ]:
X_top = X_eng

In [ ]:
X_eng["F1706_low"] = (
    X_eng["F1706"] <= 148234
).astype(int)

X_eng["F1599_low"] = (
    X_eng["F1599"] <= 90746
).astype(int)

X_eng["F1491_low"] = (
    X_eng["F1491"] <= 27000
).astype(int)

X_eng["F1921_low"] = (
    X_eng["F1921"] <= 9870
).astype(int)

In [ ]:
financial_risk_features = [
    "F1706_low",
    "F1599_low",
    "F1491_low",
    "F1921_low"
]

X_eng["financial_risk_score"] = (
    X_eng[financial_risk_features]
    .sum(axis=1)
)

In [ ]:
pd.crosstab(
    X_eng["financial_risk_score"],
    y_train,
    normalize="index"
)

F3924,0,1
financial_risk_score,,
0,0.997270,0.002730
1,0.996552,0.003448
2,0.988713,0.011287
3,0.983471,0.016529
4,0.821656,0.178344


In [ ]:
X_eng["ultimate_risk_score"] = (
      X_eng["super_risk_score"]
    + X_eng["financial_risk_score"]
)

In [ ]:
pd.crosstab(
    X_eng["ultimate_risk_score"],
    y_train,
    normalize="index"
).sort_index()

F3924,0,1
ultimate_risk_score,,
0,0.999339,0.000661
1,0.998185,0.001815
2,0.997693,0.002307
3,0.994313,0.005687
4,0.983437,0.016563
5,0.916256,0.083744
6,0.741379,0.258621
7,0.285714,0.714286


In [ ]:
X_eng["risk_x_F3898"] = (
    X_eng["super_risk_score"] *
    X_eng["F3898"]
)

X_eng["risk_x_F3920"] = (
    X_eng["super_risk_score"] *
    X_eng["F3920"]
)

X_eng["risk_x_F3921"] = (
    X_eng["super_risk_score"] *
    X_eng["F3921"]
)

In [ ]:
print(X_eng.shape)

(7265, 2936)


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_eng, y_train):

    X_tr = X_eng.iloc[train_idx]
    X_val = X_eng.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=700,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42,
        tree_method="hist"
    )

    model.fit(X_tr, y_tr)

    pred_prob = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(
        y_val,
        pred_prob
    )

    scores.append(ap)

print("Fold AP:", scores)
print("Mean AP:", np.mean(scores))

Fold AP: [np.float64(0.9194823694823693), np.float64(0.8117222165886563), np.float64(0.7943248905830953), np.float64(0.9945054945054943), np.float64(0.825318386463897)]
Mean AP: 0.8690706715247023


In [ ]:
from xgboost import XGBClassifier
import pandas as pd

model = XGBClassifier(
    n_estimators=700,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=110.77,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_eng, y_train)

imp = pd.DataFrame({
    "Feature": X_eng.columns,
    "Importance": model.feature_importances_
})

imp = imp.sort_values(
    "Importance",
    ascending=False
)

imp.head(100)

,Feature,Importance
2930,F1921_low,0.042054
1826,F2494,0.026322
2932,ultimate_risk_score,0.024385
2398,F3337,0.022939
2782,F3795,0.017626
...,...,...
1731,F2387,0.002681
766,F1058,0.002614
639,F847,0.002559
327,F379,0.002548


In [ ]:
top500 = imp.head(500)["Feature"].tolist()

X_top500 = X_eng[top500]

print(X_top500.shape)

(7265, 500)


In [ ]:
scores = []

for train_idx, val_idx in skf.split(X_top500, y_train):

    X_tr = X_top500.iloc[train_idx]
    X_val = X_top500.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=700,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    scores.append(
        average_precision_score(y_val, pred)
    )

print(scores)
print(np.mean(scores))

[np.float64(0.9305333780030134), np.float64(0.8096789686828), np.float64(0.8208032428646124), np.float64(0.9999999999999998), np.float64(0.8290935203984083)]
0.8780218219897667


In [ ]:
top300 = imp.head(300)["Feature"].tolist()

X_top300 = X_eng[top300]

print(X_top300.shape)

(7265, 300)


In [ ]:
scores = []

for train_idx, val_idx in skf.split(X_top300, y_train):

    X_tr = X_top300.iloc[train_idx]
    X_val = X_top300.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=700,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42,
        tree_method="hist"
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    scores.append(
        average_precision_score(y_val, pred)
    )

print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.9213800599252102), np.float64(0.8030345865683514), np.float64(0.8364805533745721), np.float64(0.9999999999999998), np.float64(0.8236151550254113)]
Mean AP: 0.8769020709787091


In [ ]:
X_best = X_top500

In [ ]:
model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=4,
    min_child_weight=5,
    gamma=1,
    subsample=0.9,
    colsample_bytree=0.7,
    scale_pos_weight=110.77,
    eval_metric="logloss",
    random_state=42,
    tree_method="hist"
)

In [ ]:
print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.9213800599252102), np.float64(0.8030345865683514), np.float64(0.8364805533745721), np.float64(0.9999999999999998), np.float64(0.8236151550254113)]
Mean AP: 0.8769020709787091


In [ ]:
model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=10,
    gamma=2,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=1,
    reg_lambda=5,
    scale_pos_weight=110.77,
    eval_metric="logloss",
    random_state=42,
    tree_method="hist"
)

In [ ]:
print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.9213800599252102), np.float64(0.8030345865683514), np.float64(0.8364805533745721), np.float64(0.9999999999999998), np.float64(0.8236151550254113)]
Mean AP: 0.8769020709787091


from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

print("Shape:", X_top500.shape)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top500, y_train):

    X_tr = X_top500.iloc[train_idx]
    X_val = X_top500.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=5,
        gamma=1,
        subsample=0.9,
        colsample_bytree=0.7,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42,
        tree_method="hist"
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:, 1]

    ap = average_precision_score(y_val, pred)

    scores.append(ap)

print("Experiment 1 Scores:", scores)
print("Experiment 1 Mean AP:", np.mean(scores))

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

scores = []

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for train_idx, val_idx in skf.split(X_top500, y_train):

    X_tr = X_top500.iloc[train_idx]
    X_val = X_top500.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=10,
        gamma=1,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=1,
        reg_lambda=5,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42,
        tree_method="hist"
    )

    model.fit(X_tr, y_tr)

    preds = model.predict_proba(X_val)[:, 1]

    scores.append(
        average_precision_score(y_val, preds)
    )

print(scores)
print("Mean AP:", np.mean(scores))

In [ ]:
X_best = X_top500.copy()

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_best, y_train):

    X_tr = X_best.iloc[train_idx]
    X_val = X_best.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.02,
        subsample=0.9,
        colsample_bytree=0.7,
        min_child_weight=5,
        gamma=1,
        reg_alpha=1,
        reg_lambda=3,
        scale_pos_weight=110.77,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:, 1]

    ap = average_precision_score(y_val, pred)

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.901711322865169), np.float64(0.8015101857207119), np.float64(0.8138965619489049), np.float64(0.9945054945054943), np.float64(0.8321326171166424)]
Mean AP: 0.8687512364313846


| Dataset / Model        |        Mean AP |
| ---------------------- | -------------: |
| Top 500 features       | **0.88109** 🥇 |
| Top 20 + risk features |        0.87722 |
| Top 300 features       |        0.87659 |
| Tuned XGBoost          |        0.86726 |
| Ultimate risk score    |        0.86730 |
| Experiment 1           |        0.86385 |
| Experiment 2           |        0.86086 |
| Latest experiment      |        0.85528 |


In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top500, y_train):

    X_tr = X_top500.iloc[train_idx]
    X_val = X_top500.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=1000,
        depth=6,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        verbose=0,
        random_seed=42
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:, 1]

    ap = average_precision_score(y_val, pred)

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.947234760779911), np.float64(0.8509265155588194), np.float64(0.8211635966525144), np.float64(0.9999999999999998), np.float64(0.8408241686577409)]
Mean AP: 0.8920298083297971


from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top500, y_train):

    X_tr = X_top500.iloc[train_idx]
    X_val = X_top500.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=2000,
        depth=8,
        learning_rate=0.02,
        l2_leaf_reg=5,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=0
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(y_val, pred)

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

In [ ]:
import numpy as np
cat_oof = np.zeros(len(X_top500))

In [ ]:
xgb_oof = np.zeros(len(X_top500))

In [ ]:
from sklearn.metrics import average_precision_score

blend = (
    0.7 * cat_oof +
    0.3 * xgb_oof
)

print(
    average_precision_score(
        y_train,
        blend
    )
)

0.008947006194081212


In [ ]:
print(cat_oof.min(), cat_oof.max())
print(xgb_oof.min(), xgb_oof.max())

0.0 0.0
0.0 0.0


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np

cat_oof = np.zeros(len(X_top500))

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for train_idx, val_idx in skf.split(X_top500, y_train):

    X_tr = X_top500.iloc[train_idx]
    X_val = X_top500.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]

    model = CatBoostClassifier(
        iterations=1000,
        depth=6,
        learning_rate=0.03,
        auto_class_weights="Balanced",
        verbose=0,
        random_seed=42
    )

    model.fit(X_tr, y_tr)

    cat_oof[val_idx] = model.predict_proba(X_val)[:,1]

print("CatBoost OOF AP:",
      average_precision_score(y_train, cat_oof))

CatBoost OOF AP: 0.8924864172454744


In [ ]:
print(cat_oof.min(), cat_oof.max())

3.352415495407938e-06 0.9999318083673716


In [ ]:
print(
    "XGB OOF AP:",
    average_precision_score(y_train, xgb_oof)
)

XGB OOF AP: 0.008947006194081212


In [ ]:
blend = (
    0.7 * cat_oof +
    0.3 * xgb_oof
)

print(
    "Blend AP:",
    average_precision_score(y_train, blend)
)

Blend AP: 0.8924864172454744


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

param_grid = [
    {
        "depth": 6,
        "learning_rate": 0.03,
        "l2_leaf_reg": 3,
        "iterations": 1000
    },
    {
        "depth": 8,
        "learning_rate": 0.02,
        "l2_leaf_reg": 10,
        "iterations": 3000
    },
    {
        "depth": 7,
        "learning_rate": 0.02,
        "l2_leaf_reg": 5,
        "iterations": 2000
    },
    {
        "depth": 8,
        "learning_rate": 0.01,
        "l2_leaf_reg": 20,
        "iterations": 4000
    },
    {
        "depth": 5,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3,
        "iterations": 1500
    }
]

results = []

for params in param_grid:

    fold_scores = []

    for train_idx, val_idx in skf.split(X_top500, y_train):

        X_tr = X_top500.iloc[train_idx]
        X_val = X_top500.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(
            iterations=params["iterations"],
            depth=params["depth"],
            learning_rate=params["learning_rate"],
            l2_leaf_reg=params["l2_leaf_reg"],
            loss_function="Logloss",
            eval_metric="PRAUC",
            auto_class_weights="Balanced",
            random_seed=42,
            verbose=0
        )

        model.fit(X_tr, y_tr)

        preds = model.predict_proba(X_val)[:, 1]

        ap = average_precision_score(
            y_val,
            preds
        )

        fold_scores.append(ap)

    mean_ap = np.mean(fold_scores)

    print("\n")
    print(params)
    print("Fold Scores:", fold_scores)
    print("Mean AP:", mean_ap)

    results.append({
        **params,
        "mean_ap": mean_ap
    })

results_df = pd.DataFrame(results)

print("\n========== FINAL RANKING ==========\n")

print(
    results_df.sort_values(
        "mean_ap",
        ascending=False
    )
)



{'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'iterations': 1000}
Fold Scores: [np.float64(0.947234760779911), np.float64(0.8509265155588194), np.float64(0.8211635966525144), np.float64(0.9999999999999998), np.float64(0.8408241686577409)]
Mean AP: 0.8920298083297971


In [ ]:
top700 = importance_df.head(700)["Feature"]

X_top700 = X_eng[top700].copy()

print(X_top700.shape)

(7265, 700)


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top700, y_train):

    X_tr = X_top700.iloc[train_idx]
    X_val = X_top700.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=1000,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=3,
        auto_class_weights="Balanced",
        eval_metric="PRAUC",
        random_seed=42,
        verbose=0
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(
        y_val,
        pred
    )

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.9774161735700195), np.float64(0.8614249298459823), np.float64(0.8065612584480508), np.float64(0.9599272522349442), np.float64(0.8623521183644117)]
Mean AP: 0.8935363464926818


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top700, y_train):

    X_tr = X_top700.iloc[train_idx]
    X_val = X_top700.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
    iterations=1200,
    depth=8,
    learning_rate=0.03,
    l2_leaf_reg=5,
    auto_class_weights="Balanced",
    eval_metric="PRAUC",
    random_seed=42,
    verbose=0
)

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(
        y_val,
        pred
    )

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.9465478965478964), np.float64(0.8398147184344747), np.float64(0.7447337331058261), np.float64(0.9645011850894201), np.float64(0.8222641339979341)]
Mean AP: 0.8635723334351102


exp 5

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top700, y_train):

    X_tr = X_top700.iloc[train_idx]
    X_val = X_top700.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.015,
    l2_leaf_reg=3,
    auto_class_weights="Balanced",
    eval_metric="PRAUC",
    random_seed=42,
    verbose=0
)
    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(
        y_val,
        pred
    )

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

[np.float64(0.9885883347421807), np.float64(0.8457011013887074), np.float64(0.8160882595665202), np.float64(0.9945054945054943), np.float64(0.8849396681749621)]
Mean AP: 0.9059645716755729


In [ ]:
top1000 = importance_df.head(1000)["Feature"]

X_top1000 = X_eng[top1000].copy()

print(X_top1000.shape)

(7265, 1000)


exp 6

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top1000, y_train):

    X_tr = X_top1000.iloc[train_idx]
    X_val = X_top1000.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=3,
    auto_class_weights="Balanced",
    eval_metric="PRAUC",
    random_seed=42,
    verbose=0
)
    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(
        y_val,
        pred
    )

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

In [ ]:
top200 = importance_df.head(200)["Feature"]

X_top200 = X_eng[top200].copy()

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top200, y_train):

    X_tr = X_top200.iloc[train_idx]
    X_val = X_top200.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.015,
    l2_leaf_reg=3,
    auto_class_weights="Balanced",
    eval_metric="PRAUC",
    random_seed=42,
    verbose=0
)
    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(
        y_val,
        pred
    )

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

neeche exp 6

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import numpy as np

scores = []

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for tr_idx, val_idx in skf.split(X_top700, y_train):

    X_tr = X_top700.iloc[tr_idx]
    X_val = X_top700.iloc[val_idx]

    y_tr = y_train.iloc[tr_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=2500,
        depth=8,
        learning_rate=0.02,
        l2_leaf_reg=5,
        loss_function='Logloss',
        eval_metric='PRAUC',
        auto_class_weights='Balanced',
        verbose=0,
        random_seed=42
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    scores.append(
        average_precision_score(
            y_val,
            pred
        )
    )

print(scores)
print("Mean AP:", np.mean(scores))

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_eng.columns,
    "Importance": model.feature_importances_
})

top1000 = (
    importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .head(1000)
)

selected_features = top1000["Feature"].tolist()

X_top1000 = X_eng[selected_features]

print(X_top1000.shape)

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import numpy as np

scores = []

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for tr_idx, val_idx in skf.split(X_top1000, y_train):

    X_tr = X_top1000.iloc[tr_idx]
    X_val = X_top1000.iloc[val_idx]

    y_tr = y_train.iloc[tr_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=2500,
        depth=8,
        learning_rate=0.02,
        l2_leaf_reg=5,
        loss_function='Logloss',
        eval_metric='PRAUC',
        auto_class_weights='Balanced',
        verbose=0,
        random_seed=42
    )

    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    scores.append(
        average_precision_score(
            y_val,
            pred
        )
    )

print(scores)
print("Mean AP:", np.mean(scores))

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

print(neg)
print(pos)

scale_pos_weight = neg / pos

print(scale_pos_weight)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in skf.split(X_top700, y_train):

    X_tr = X_top700.iloc[train_idx]
    X_val = X_top700.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
    iterations=2500,
    depth=8,
    learning_rate=0.02,
    l2_leaf_reg=5,
    scale_pos_weight=scale_pos_weight,
    eval_metric="PRAUC",
    verbose=0
)
    model.fit(X_tr, y_tr)

    pred = model.predict_proba(X_val)[:,1]

    ap = average_precision_score(
        y_val,
        pred
    )

    scores.append(ap)

print(scores)
print("Mean AP:", np.mean(scores))

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import numpy as np

feature_counts = [1000, 1200]

results = {}

for n_feats in feature_counts:

    selected_features = (
        importance_df
        .sort_values("Importance", ascending=False)
        .head(n_feats)["Feature"]
        .tolist()
    )

    X_temp = X_eng[selected_features]

    scores = []

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for train_idx, val_idx in skf.split(X_temp, y_train):

        X_tr = X_temp.iloc[train_idx]
        X_val = X_temp.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(
            iterations=2000,
            depth=6,
            learning_rate=0.015,
            l2_leaf_reg=3,
            auto_class_weights="Balanced",
            eval_metric="PRAUC",
            random_seed=42,
            verbose=0
        )

        model.fit(X_tr, y_tr)

        pred = model.predict_proba(X_val)[:,1]

        scores.append(
            average_precision_score(
                y_val,
                pred
            )
        )

    mean_ap = np.mean(scores)

    results[n_feats] = mean_ap

    print(
        f"Features={n_feats} | AP={mean_ap:.6f}"
    )

print(results)